In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Fungsi pembantu untuk memastikan folder ada
def safe_save_path(path):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)
        print(f"Folder dibuat: {path}")
    return path

PROJECT_ROOT = '/content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels'
safe_save_path(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")

Mounted at /content/drive
Working directory: /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels


In [ ]:
import json
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from tqdm import tqdm
import random
import glob

random.seed(42)
np.random.seed(42)

print("Libraries imported")

Libraries imported


In [ ]:
# Paths
DATASET_PATH = os.path.join(PROJECT_ROOT, 'dataset')
DATASET_ROOT = os.path.join(DATASET_PATH, 'RDD_SPLIT')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'analysis_results')
OUTPUT_ROOT = os.path.join(PROJECT_ROOT, 'processed_dataset')

safe_save_path(RESULTS_DIR)
safe_save_path(OUTPUT_ROOT)

# Create output structure
os.makedirs(OUTPUT_ROOT, exist_ok=True)
for split in ['train', 'val', 'test']:
    for subdir in ['images', 'labels', 'severity']:
        os.makedirs(os.path.join(OUTPUT_ROOT, split, subdir), exist_ok=True)

# Class config
CLASS_NAMES = {
    0: 'Longitudinal Crack',
    1: 'Transverse Crack',
    2: 'Alligator Crack',
    4: 'Pothole'
}

USED_CLASSES = [0, 1, 2, 4]
EXCLUDE_CLASS = 3
TARGET_TRAIN_IMAGES = 10000

print("Configuration loaded")
print(f"Output root: {OUTPUT_ROOT}")

Configuration loaded
Output root: /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/processed_dataset


In [ ]:
threshold_file = os.path.join(RESULTS_DIR, 'severity_thresholds.json')

with open(threshold_file, 'r') as f:
    SEVERITY_THRESHOLDS = json.load(f)

# Convert keys to int
SEVERITY_THRESHOLDS = {int(k): v for k, v in SEVERITY_THRESHOLDS.items()}

print("Severity thresholds loaded:")
for class_id in USED_CLASSES:
    low = SEVERITY_THRESHOLDS[class_id]['low_threshold']
    high = SEVERITY_THRESHOLDS[class_id]['high_threshold']
    print(f"  Class {class_id}: Rendah < {low:.4f}, Sedang < {high:.4f}, Tinggi >= {high:.4f}")

Severity thresholds loaded:
  Class 0: Rendah < 0.0500, Sedang < 0.1500, Tinggi >= 0.1500
  Class 1: Rendah < 0.0500, Sedang < 0.1500, Tinggi >= 0.1500
  Class 2: Rendah < 0.0500, Sedang < 0.1500, Tinggi >= 0.1500
  Class 4: Rendah < 0.0500, Sedang < 0.1500, Tinggi >= 0.1500


In [ ]:
def parse_yolo_label(label_path):
    """Parse and filter YOLO labels"""
    bboxes = []
    if not os.path.exists(label_path):
        return bboxes

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            if class_id == EXCLUDE_CLASS:
                continue

            bboxes.append({
                'class_id': class_id,
                'x_center': float(parts[1]),
                'y_center': float(parts[2]),
                'width': float(parts[3]),
                'height': float(parts[4]),
                'area_ratio': float(parts[3]) * float(parts[4])
            })

    return bboxes

def assign_severity(bbox):
    """Assign severity based on area ratio"""
    class_id = bbox['class_id']
    area_ratio = bbox['area_ratio']
    thresholds = SEVERITY_THRESHOLDS[class_id]

    if area_ratio < thresholds['low_threshold']:
        return 0  # Rendah
    elif area_ratio < thresholds['high_threshold']:
        return 1  # Sedang
    else:
        return 2  # Tinggi

def get_image_classes(label_path):
    """Get classes in image"""
    bboxes = parse_yolo_label(label_path)
    return set([bbox['class_id'] for bbox in bboxes])

print("Helper functions defined")

Helper functions defined


In [ ]:
print("Scanning training images...")

train_images_dir = os.path.join(DATASET_ROOT, 'train', 'images')
train_labels_dir = os.path.join(DATASET_ROOT, 'train', 'labels')

images_by_class = defaultdict(list)
all_train_images = [f for f in os.listdir(train_images_dir) if f.endswith(('.jpg', '.png'))]

for img_name in tqdm(all_train_images):
    label_name = os.path.splitext(img_name)[0] + '.txt'
    label_path = os.path.join(train_labels_dir, label_name)

    classes = get_image_classes(label_path)
    for class_id in classes:
        if class_id in USED_CLASSES:
            images_by_class[class_id].append(img_name)

print("\nImages per class:")
for class_id in USED_CLASSES:
    print(f"  Class {class_id}: {len(images_by_class[class_id])} images")

Scanning training images...


100%|██████████| 26869/26869 [1:46:19<00:00,  4.21it/s]


Images per class:
  Class 0: 9457 images
  Class 1: 5433 images
  Class 2: 5976 images
  Class 4: 2599 images


In [ ]:
# Calculate proportional samples
total_available = sum(len(images_by_class[c]) for c in USED_CLASSES)
samples_per_class = {}

for class_id in USED_CLASSES:
    proportion = len(images_by_class[class_id]) / total_available
    samples_per_class[class_id] = int(TARGET_TRAIN_IMAGES * proportion)

print("Target samples per class:")
for class_id in USED_CLASSES:
    print(f"  Class {class_id}: {samples_per_class[class_id]} images")

# Perform sampling
selected_images = set()

for class_id in USED_CLASSES:
    available = images_by_class[class_id]
    target = samples_per_class[class_id]
    sampled = random.sample(available, min(target, len(available)))
    selected_images.update(sampled)

selected_images = list(selected_images)
print(f"\nTotal unique images selected: {len(selected_images)}")

Target samples per class:
  Class 0: 4030 images
  Class 1: 2315 images
  Class 2: 2546 images
  Class 4: 1107 images

Total unique images selected: 8690


In [ ]:
print("\nCopying sampled train data...")

output_train_img = os.path.join(OUTPUT_ROOT, 'train', 'images')
output_train_lbl = os.path.join(OUTPUT_ROOT, 'train', 'labels')

copied = 0
for img_name in tqdm(selected_images):
    src_img = os.path.join(train_images_dir, img_name)
    label_name = os.path.splitext(img_name)[0] + '.txt'
    src_label = os.path.join(train_labels_dir, label_name)

    dst_img = os.path.join(output_train_img, img_name)
    dst_label = os.path.join(output_train_lbl, label_name)

    if not os.path.exists(src_img):
        continue

    shutil.copy2(src_img, dst_img)

    bboxes = parse_yolo_label(src_label)
    with open(dst_label, 'w') as f:
        for bbox in bboxes:
            f.write(f"{bbox['class_id']} {bbox['x_center']} {bbox['y_center']} "
                   f"{bbox['width']} {bbox['height']}\n")

    copied += 1

print(f"Copied {copied} images to train set")


Copying sampled train data...


100%|██████████| 8690/8690 [54:44<00:00,  2.65it/s]

Copied 8690 images to train set


In [ ]:
def copy_split(split_name):
    print(f"\nCopying {split_name} set...")

    src_img_dir = os.path.join(DATASET_ROOT, split_name, 'images')
    src_lbl_dir = os.path.join(DATASET_ROOT, split_name, 'labels')
    dst_img_dir = os.path.join(OUTPUT_ROOT, split_name, 'images')
    dst_lbl_dir = os.path.join(OUTPUT_ROOT, split_name, 'labels')

    all_images = [f for f in os.listdir(src_img_dir) if f.endswith(('.jpg', '.png'))]

    for img_name in tqdm(all_images):
        src_img = os.path.join(src_img_dir, img_name)
        label_name = os.path.splitext(img_name)[0] + '.txt'
        src_label = os.path.join(src_lbl_dir, label_name)

        dst_img = os.path.join(dst_img_dir, img_name)
        dst_label = os.path.join(dst_lbl_dir, label_name)

        shutil.copy2(src_img, dst_img)

        bboxes = parse_yolo_label(src_label)
        with open(dst_label, 'w') as f:
            for bbox in bboxes:
                f.write(f"{bbox['class_id']} {bbox['x_center']} {bbox['y_center']} "
                       f"{bbox['width']} {bbox['height']}\n")

    print(f"  {len(all_images)} images copied")

copy_split('val')
copy_split('test')


Copying val set...


100%|██████████| 5758/5758 [28:00<00:00,  3.43it/s]


  5758 images copied

Copying test set...


100%|██████████| 5758/5758 [27:40<00:00,  3.47it/s]

  5758 images copied


In [ ]:
def generate_severity_for_split(split_name):
    print(f"\nGenerating severity for {split_name}...")

    img_dir = os.path.join(OUTPUT_ROOT, split_name, 'images')
    lbl_dir = os.path.join(OUTPUT_ROOT, split_name, 'labels')
    sev_dir = os.path.join(OUTPUT_ROOT, split_name, 'severity')

    all_images = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
    severity_stats = Counter()

    for img_name in tqdm(all_images):
        label_name = os.path.splitext(img_name)[0] + '.txt'
        label_path = os.path.join(lbl_dir, label_name)

        bboxes = parse_yolo_label(label_path)
        severity_data = []

        for idx, bbox in enumerate(bboxes):
            severity = assign_severity(bbox)
            severity_stats[severity] += 1

            severity_data.append({
                'bbox_id': idx,
                'class_id': bbox['class_id'],
                'class_name': CLASS_NAMES[bbox['class_id']],
                'severity': severity,
                'severity_label': ['Rendah', 'Sedang', 'Tinggi'][severity],
                'area_ratio': bbox['area_ratio'],
                'bbox': [bbox['x_center'], bbox['y_center'], bbox['width'], bbox['height']]
            })

        sev_file = os.path.join(sev_dir, os.path.splitext(img_name)[0] + '.json')
        with open(sev_file, 'w') as f:
            json.dump({
                'image_name': img_name,
                'num_damages': len(severity_data),
                'annotations': severity_data
            }, f, indent=2)

    total = sum(severity_stats.values())
    print(f"  Severity distribution:")
    for sev in [0, 1, 2]:
        count = severity_stats[sev]
        pct = (count / total * 100) if total > 0 else 0
        print(f"    {['Rendah', 'Sedang', 'Tinggi'][sev]}: {count} ({pct:.1f}%)")

    return severity_stats

stats = {}
for split in ['train', 'val', 'test']:
    stats[split] = generate_severity_for_split(split)

In [ ]:
# Create data.yaml
data_yaml = f"""path: {OUTPUT_ROOT}
train: train/images
val: val/images
test: test/images

nc: 4
names:
  0: Longitudinal_Crack
  1: Transverse_Crack
  2: Alligator_Crack
  4: Pothole
"""

with open(os.path.join(OUTPUT_ROOT, 'data.yaml'), 'w') as f:
    f.write(data_yaml.strip())

print("data.yaml created")

# Summary
print("\n" + "="*60)
print("FINAL DATASET SUMMARY")
print("="*60)

for split in ['train', 'val', 'test']:
    img_dir = os.path.join(OUTPUT_ROOT, split, 'images')
    num_imgs = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])
    print(f"\n{split.upper()}: {num_imgs} images")

print(f"\nProcessed dataset saved to:\n   {OUTPUT_ROOT}")

data.yaml created

FINAL DATASET SUMMARY

TRAIN: 8690 images

VAL: 5758 images

TEST: 5758 images

Processed dataset saved to:
   /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/processed_dataset


In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("Semua data telah disinkronkan dan aman di Drive.")

Semua data telah disinkronkan dan aman di Drive.
